In [1]:
%load_ext autoreload
%autoreload 2
%reset -f

In [2]:
from locallib.picarrodb import *
from locallib.query import *
from data_extractor import ExtractionSpec, extract_multiple_data
from dotenv import load_dotenv
from apps_dal_dynamo.dynamodal import DynamoDalBase

import datetime
import time
import os

/home/sandbox/personal-repos/.venv/lib/python3.9/site-packages/geopandas/_compat.py:154: UserWarning: The Shapely GEOS version (3.10.3-CAPI-1.16.1) is incompatible with the GEOS version PyGEOS was compiled with (3.10.1-CAPI-1.16.0). Conversions between both will be slow.
  set_use_pygeos()


EU1_Conn created successfully
EU2_Conn created successfully
DataHub_Conn created successfully
US_Conn created successfully
EU1_PROD_Conn created successfully
EU2_PROD_Conn created successfully


In [3]:
import boto3
import dotenv
import io
import pandas as pd 
import os
import time

from athena_query import run_athena_query

dotenv.load_dotenv()


True

In [4]:
#Athena setup
# Get AWS setup
AWS_ATHENA_US_PROFILE = os.getenv('AWS_ATHENA_US_PROFILE')
AWS_ATHENA_DYNAMOEXPORTS_WG = os.getenv('AWS_ATHENA_DYNAMOEXPORTS_WG')
AWS_ATHENA_DYNAMOEXPORTS_DB=os.getenv('AWS_ATHENA_DYNAMOEXPORTS_DB')
AWS_ATHENA_DYNAMOEXPORTS_REGION=os.getenv('AWS_ATHENA_DYNAMOEXPORTS_REGION')
AWS_ATHENA_HANDHELD_WG = os.getenv('AWS_ATHENA_HANDHELD_WG')
AWS_ATHENA_HANDHELD_DB = os.getenv('AWS_ATHENA_HANDHELD_DB')
AWS_ATHENA_HANDHELD_REGION = os.getenv('AWS_ATHENA_HANDHELD_REGION')
for var in [
    AWS_ATHENA_US_PROFILE,
    AWS_ATHENA_DYNAMOEXPORTS_WG,
    AWS_ATHENA_DYNAMOEXPORTS_DB,
    AWS_ATHENA_DYNAMOEXPORTS_REGION,
    AWS_ATHENA_HANDHELD_WG,
    AWS_ATHENA_HANDHELD_DB,
    AWS_ATHENA_HANDHELD_REGION,
]:
    assert var is not None, f"Variable {var} is not set"
available = boto3.Session().available_profiles
print("AWS profiles on this machine:", available or "(none)")
print("Expected profile:", AWS_ATHENA_US_PROFILE)

if AWS_ATHENA_US_PROFILE not in available:
    raise RuntimeError(
        f"AWS profile '{AWS_ATHENA_US_PROFILE}' not found.\n"
        "Run: bash setup_athena_aws.sh\n"
        "You need your org's AWS SSO start URL (AWS access portal URL)."
    )
print("Profile found — OK")

session_dynamoexports = boto3.Session(
  profile_name=AWS_ATHENA_US_PROFILE, 
  region_name=AWS_ATHENA_DYNAMOEXPORTS_REGION
)
athena_dynamoexports = session_dynamoexports.client('athena')
s3_dynamoexports = session_dynamoexports.client('s3')

AWS profiles on this machine: ['default', 'Data-Analytics-Athena-User-580855453261']
Expected profile: Data-Analytics-Athena-User-580855453261
Profile found — OK


In [5]:
# Set analyzer_id and range of months to fetch
analyzer_id_lower = '72febf73-edcc-c570-171c-39f3bf42f9d2'
analyzer_id_upper = '72febf73-edcc-c570-171c-39f3bf42f9d2'.upper()
start_date = "03-01-2024"
end_date = "09-01-2024"

In [21]:
q = f"""SELECT S.Id as SurveyId, 
        S.StartEpoch as StartEpoch, 
        S.EndEpoch as EndEpoch, 
        (SELECT Description FROM SurveyorUnit SU WHERE SU.Id = S.SurveyorUnitId) AS SurveyorUnit,
        (SELECT COUNT(P.Id) FROM Peak P WHERE P.SurveyId = S.Id) as PeakCount,
        MONTH(StartDateTime) as Month, 
        YEAR(StartDateTime) as Year 
        FROM Survey S 
        WHERE AnalyzerId = '{analyzer_id_lower}' 
        AND StartDateTime >= '{start_date}' 
        AND StartDateTime <= '{end_date}' """
surveys = Query(q).execute(US_Conn)

In [22]:
surveys

,SurveyId,StartEpoch,EndEpoch,SurveyorUnit,PeakCount,Month,Year
0,A9AE6810-39C6-8288-C5D1-3A121C49125C,1.713862e+09,1.713873e+09,B34654 - Oakland (old unused),20,4,2024
1,350698EE-1D96-81AA-F448-3A121BD6EF74,1.713854e+09,1.713861e+09,B34654 - Oakland (old unused),9,4,2024
2,E717B2D9-83A8-63FC-73E3-3A1217349E27,1.713776e+09,1.713786e+09,B34654 - Oakland (old unused),51,4,2024
3,7B1C52A2-9751-FEE2-9DB3-3A121724B788,1.713775e+09,1.713776e+09,B34654 - Oakland (old unused),3,4,2024
4,9DC94B95-3C8C-22D5-8C42-3A1215A0AD51,1.713750e+09,1.713775e+09,B34654 - Oakland (old unused),113,4,2024
...,...,...,...,...,...,...,...
224,8C330E69-EBC9-648B-4BA5-3A122AA07426,1.714102e+09,1.714131e+09,B41107 - Oakland,63,4,2024
225,07F33BA7-F581-2C94-2405-3A1226C3EA85,1.714037e+09,1.714047e+09,B41107 - Oakland,16,4,2024
226,61D9496D-7B2A-747E-E00F-3A12259C71A8,1.714018e+09,1.714037e+09,B41107 - Oakland,36,4,2024
227,5A6EBA8B-D16C-37D6-24E9-3A1220AC6153,1.713935e+09,1.713961e+09,B41107 - Oakland,61,4,2024


In [ ]:
results = []
for month in surveys['Month'].unique():
    print('Getting Athena data for month', month)
    query_dynamoexports = f"""
    SELECT *
    FROM surveyorproduction_measurement_parquet
    WHERE analyzer_id = '{analyzer_id_lower}'
    AND YEAR = 2024
    AND MONTH = {month}
    """
    out_loc_dynamoexports = run_athena_query(
        query=query_dynamoexports, 
        client=athena_dynamoexports,
        db=AWS_ATHENA_DYNAMOEXPORTS_DB, 
        wg=AWS_ATHENA_DYNAMOEXPORTS_WG
    )
    if out_loc_dynamoexports is None:
        print(f"Athena query failed")
        continue
    bucket_dynamoexports, _, key_dynamoexports = out_loc_dynamoexports[len("s3://"):].partition("/")
    obj_dynamoexports = s3_dynamoexports.get_object(Bucket=bucket_dynamoexports, Key=key_dynamoexports)
    # Read CSV into DataFrame and append
    athena_month = pd.read_csv(io.BytesIO(obj_dynamoexports['Body'].read()))
    surveys_month = surveys[surveys['Month'] == month]
    print('Number of surveys in month', month, len(surveys_month))
    for idx, survey in surveys_month.iterrows():
        print('Processing survey', survey['SurveyId'])
        data_survey = athena_month[(athena_month['epoch_time'] >= survey['StartEpoch'])&(athena_month['epoch_time'] <= survey['EndEpoch'])]
        averageFlowRate = data_survey['mobile_flow_rate'].mean()
        surveyId = survey['SurveyId']
        startEpoch = survey['StartEpoch']
        endEpoch = survey['EndEpoch']
        surveyorUnit = survey['SurveyorUnit']
        peakCount = survey['PeakCount']
        rawDurationMinutes = (endEpoch - startEpoch) / 60
        peakMinutes = peakCount / rawDurationMinutes
   
        results.append({'SurveyId': surveyId, 'AverageFlowRate': averageFlowRate, 'StartEpoch': startEpoch, 'EndEpoch': endEpoch, 'SurveyorUnit': surveyorUnit, 'PeakCount': peakCount, 'RawDurationMinutes': rawDurationMinutes, 'PeakMinutes': peakMinutes})

# Create a dataframe from the results list
df = pd.DataFrame(results)


Getting Athena data for month 8
Number of surveys in month 8 50
Processing survey E7FF94D9-1767-FF9F-7274-3A14B4E1C0B9
Processing survey 299ED000-5ADA-95D7-B7C4-3A14B38E82F4
Processing survey 3FE67A7C-E498-701F-A7BC-3A14AF7BC043
Processing survey BB7ABC22-30FE-CCD1-0BA3-3A14AE610DFD
Processing survey 5BE8E4E8-763F-9092-6F93-3A14A9F50693
Processing survey DBC94CE6-864F-1C9B-925C-3A14A9EA5FE3
Processing survey 927B6D5A-ABF9-B885-FA30-3A14A9D678B7
Processing survey D0EE82DE-18D1-BFCD-2296-3A14A58D6E8C
Processing survey 9C10AF08-78AF-5028-D68E-3A14A5873B0D
Processing survey 231C1E4B-EAF0-3A32-25FB-3A14A4122328
Processing survey 0F0A11FE-882F-AA41-F530-3A14910FEC5D
Processing survey AB576A7E-030B-502C-D3F0-3A14904ADD79
Processing survey 0455D81B-0E97-F239-2CA9-3A1490475967
Processing survey 203A2067-9974-F96B-D71E-3A148F7D2339
Processing survey 038C39EB-3435-812A-3539-3A148BBAB403
Processing survey 22FF4C6F-F607-3524-FEF0-3A148A4F27C3
Processing survey 10987BB3-C6BF-BB2E-4448-3A14862B6D8C
P

In [25]:
df_with_dates = df.copy()
df_with_dates['StartDateTime'] = pd.to_datetime(df_with_dates['StartEpoch'], unit='s')
df_with_dates['EndDateTime'] = pd.to_datetime(df_with_dates['EndEpoch'], unit='s')
df_with_dates['CaseId'] = 'US_01'